In [10]:
def create_table():
    conn = mc.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    # 가이드 문서의 필드 명세에 맞춘 테이블 생성 쿼리
    create_query = """
    CREATE TABLE IF NOT EXISTS speed_pattern_monthly (
        id BIGINT AUTO_INCREMENT PRIMARY KEY,
        statDate VARCHAR(6) NOT NULL,
        roadName VARCHAR(100),
        direction VARCHAR(20),
        sectionName VARCHAR(200),
        monthStatValue INT,
        mon INT, tue INT, wed INT, thu INT, fri INT, sat INT, sun INT,
        UNIQUE KEY uniq_pattern (statDate, roadName, direction, sectionName)
    );
    """
    try:
        cursor.execute(create_query)
        conn.commit()
        print("[INFO] speed_pattern_monthly 테이블 생성 완료")
    except Exception as e:
        print(f"[ERROR] 테이블 생성 실패: {e}")
    finally:
        cursor.close()
        conn.close()


create_table()

[INFO] speed_pattern_monthly 테이블 생성 완료


In [1]:
import requests
import os
import mysql.connector as mc
from datetime import datetime
import dotenv
import time

dotenv.load_dotenv()


BASE_URL = 'http://apis.data.go.kr/6280000/ICRoadStat_v2/STAT-PatternSpeed_MM_Section'
SERVICE_KEY = os.getenv('PUBLIC_DATA_API_KEY')
MAX_PAGE_SIZE = 5000

DB_CONFIG = {
    'host': os.getenv('DB_HOST'),
    'port': int(os.getenv('DB_PORT', 3306)),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'database': os.getenv('DB_DATABASE'),
    'autocommit': True,
}

# 수집할 주요 도로 리스트 
ROAD_NAMES = [
    '경인로', '봉오대로', '아암대로', '인주대로', '중봉대로', 
    '경원대로', '남동대로', '무네미로', '비류대로', '드림로',
    '서해대로', '수봉로', '장제로', '백범로', '호구포로'
]

columns = [
    "statDate", "roadName", "direction", "sectionName", 
    "monthStatValue", "mon", "tue", "wed", "thu", "fri", "sat", "sun"
]
col_str = ", ".join(columns)
placeholders = ", ".join(["%s"] * len(columns))
insert_query = f"INSERT IGNORE INTO speed_pattern_monthly ({col_str}) VALUES ({placeholders})"

def get_monthly_road_data(ym, road_name):
    """특정 월, 특정 도로의 데이터 호출"""
    params = {
        'serviceKey': SERVICE_KEY,
        'pageNo': 1,
        'numOfRows': MAX_PAGE_SIZE,
        'YM': ym,
        'roadName': road_name,
        '_type': 'json'
    }
    try:
        response = requests.get(BASE_URL, params=params, timeout=15)
        return response.json()
    except:
        return None

def extract_items(json_data):
    try:
        if not json_data: return []
        
      
        response = json_data.get('response', {})
        body = response.get('body', {})
        items_entry = body.get('items', []) 
        
       
        if isinstance(items_entry, dict):
            items = items_entry.get('item', [])
        else:
            
            items = items_entry

        
        if not items: return []
        
       
        if isinstance(items, dict):
            items = [items]
            
        processed = []
        for item in items:
            if not isinstance(item, dict): continue 
            
            row = []
            for col in columns:
                val = item.get(col, None)
                
              
                if col in ["monthStatValue", "mon", "tue", "wed", "thu", "fri", "sat", "sun"]:
                    try:
                        row.append(int(float(val)) if val is not None else 0)
                    except:
                        row.append(0)
                else:
                    row.append(str(val) if val is not None else "")
            processed.append(tuple(row))
        return processed
    except Exception as e:
        print(f"\n[파싱 에러 상세] {e}")
        return []
    
def main():
    conn = mc.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    # 기간 설정: 202301 ~ 202603
    years = [2023, 2024, 2025, 2026]
    total_count = 0

    for road in ROAD_NAMES:
        print(f"\n[도로 시작] {road} 수집 중...")
        for year in years:
            for month in range(1, 13):
                if year == 2026 and month > 3: break
                
                ym = f"{year}{str(month).zfill(2)}"
                data = get_monthly_road_data(ym, road)
                rows = extract_items(data)
                
                if rows:
                    cursor.executemany(insert_query, rows)
                    total_count += len(rows)
                    print(f"  - {ym}: {len(rows)}건 저장", end="\r")
                
                time.sleep(0.2) 

    print(f"\n\n[최종 결과] 총 {total_count}건의 데이터가 저장되었습니다.")
    cursor.close()
    conn.close()

if __name__ == "__main__":
    main()


[도로 시작] 경인로 수집 중...
  - 202603: 26건 저장
[도로 시작] 봉오대로 수집 중...
  - 202603: 20건 저장
[도로 시작] 아암대로 수집 중...
  - 202603: 20건 저장
[도로 시작] 인주대로 수집 중...
  - 202603: 24건 저장
[도로 시작] 중봉대로 수집 중...
  - 202603: 18건 저장
[도로 시작] 경원대로 수집 중...
  - 202603: 38건 저장
[도로 시작] 남동대로 수집 중...
  - 202603: 20건 저장
[도로 시작] 무네미로 수집 중...
  - 202603: 8건 저장
[도로 시작] 비류대로 수집 중...
  - 202603: 18건 저장
[도로 시작] 드림로 수집 중...
  - 202603: 6건 저장
[도로 시작] 서해대로 수집 중...
  - 202603: 12건 저장
[도로 시작] 수봉로 수집 중...

[도로 시작] 장제로 수집 중...
  - 202603: 18건 저장
[도로 시작] 백범로 수집 중...
  - 202603: 24건 저장
[도로 시작] 호구포로 수집 중...
  - 202603: 18건 저장

[최종 결과] 총 10290건의 데이터가 저장되었습니다.
